In [1]:
# 1. 필요한 라이브러리 불러오기 및 기본 경로 설정

from pathlib import Path
import shutil
import random
from collections import Counter
from PIL import Image

ROOT = Path("/data1/workspaces/tp2026/scooter-anomaly-detection")

RAW_IMG = ROOT / "data/raw/images"
RAW_LBL = ROOT / "data/raw/labels"

PROCESSED = ROOT / "data/processed"
YOLO = ROOT / "data/yolo"

CLASS_MAP = {
    "normal": 0,
    "abnormal": 1
}

In [2]:
# 2. processed, yolo 폴더 초기화

for path in [PROCESSED, YOLO]:
    if path.exists():
        shutil.rmtree(path)

for status in ["normal", "abnormal"]:
    (PROCESSED / "images" / status).mkdir(parents=True, exist_ok=True)
    (PROCESSED / "labels" / status).mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    (YOLO / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO / "labels" / split).mkdir(parents=True, exist_ok=True)

In [3]:
# 3. 이미지 파일명에서 base name 생성
# 라벨 txt를 찾고, processed에서 이미지/라벨 이름을 통일하기 위해 사용
# 예: normal_001.jpg.jpeg -> normal_001

def get_base_name(filename):
    name = filename
    for ext in [".jpeg", ".jpg", ".png", ".JPEG", ".JPG", ".PNG"]:
        name = name.replace(ext, "")
    return name

In [4]:
# 4. 라벨 txt의 class 번호 변경
# normal은 0, abnormal은 1로 저장
# bbox 좌표는 그대로 유지

def rewrite_label_class(src_label, dst_label, class_id):
    lines = src_label.read_text().strip().splitlines()
    new_lines = []

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        parts[0] = str(class_id)
        new_lines.append(" ".join(parts[:5]))

    dst_label.write_text("\n".join(new_lines) + "\n")

In [5]:
# 5. raw 폴더의 이미지와 라벨을 매칭해서 processed 폴더로 저장
# 이미지 format은 실제 JPEG로 변환하고 확장자는 .jpg로 통일
# 이미지 이름과 라벨 이름도 동일하게 맞춤
# normal 라벨은 0, abnormal 라벨은 1로 변환

processed_items = []
missing_labels = []
image_errors = []

for status, class_id in CLASS_MAP.items():
    img_root = RAW_IMG / status
    lbl_root = RAW_LBL / status

    for img_path in img_root.rglob("*"):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        rel = img_path.relative_to(img_root)
        person = rel.parent.name

        base_name = get_base_name(img_path.name)
        label_path = lbl_root / rel.parent / f"{base_name}.txt"

        if not label_path.exists():
            missing_labels.append((img_path, label_path))
            continue

        new_name = f"{status}_{person}_{base_name}"

        img_dst = PROCESSED / "images" / status / f"{new_name}.jpg"
        lbl_dst = PROCESSED / "labels" / status / f"{new_name}.txt"

        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img.save(img_dst, format="JPEG", quality=95)
        except Exception as e:
            image_errors.append((img_path, e))
            continue

        rewrite_label_class(label_path, lbl_dst, class_id)

        processed_items.append({
            "image": img_dst,
            "label": lbl_dst,
            "status": status,
            "class_id": class_id
        })

print("processed 생성 완료")
print("사용 가능 데이터:", len(processed_items))
print("라벨 없는 이미지:", len(missing_labels))
print("이미지 변환 실패:", len(image_errors))

for img, expected_lbl in missing_labels[:20]:
    print("라벨 없음:", img)
    print("찾은 경로:", expected_lbl)

for img, err in image_errors[:20]:
    print("이미지 변환 실패:", img)
    print("에러:", err)

processed 생성 완료
사용 가능 데이터: 1103
라벨 없는 이미지: 2
이미지 변환 실패: 0
라벨 없음: /data1/workspaces/tp2026/scooter-anomaly-detection/data/raw/images/normal/haikal/normal_haikal_003.jpg.jpeg
찾은 경로: /data1/workspaces/tp2026/scooter-anomaly-detection/data/raw/labels/normal/haikal/normal_haikal_003.txt
라벨 없음: /data1/workspaces/tp2026/scooter-anomaly-detection/data/raw/images/normal/ikwan/normal_Ikhwan_103.jpg
찾은 경로: /data1/workspaces/tp2026/scooter-anomaly-detection/data/raw/labels/normal/ikwan/normal_Ikhwan_103.txt


In [6]:
# 6. processed 폴더에 저장된 normal/abnormal 데이터 개수 확인

print(Counter([item["status"] for item in processed_items]))

for status in ["normal", "abnormal"]:
    img_count = len(list((PROCESSED / "images" / status).glob("*.jpg")))
    lbl_count = len(list((PROCESSED / "labels" / status).glob("*.txt")))
    print(status, "images:", img_count, "labels:", lbl_count)

Counter({'normal': 596, 'abnormal': 507})
normal images: 596 labels: 596
abnormal images: 507 labels: 507


In [7]:
# 7. processed 라벨의 class 번호 확인
# normal은 0만, abnormal은 1만 나와야 정상

for status in ["normal", "abnormal"]:
    counts = Counter()

    for txt in (PROCESSED / "labels" / status).glob("*.txt"):
        for line in txt.read_text().strip().splitlines():
            if line.strip():
                counts[line.split()[0]] += 1

    print(status, counts)

normal Counter({'0': 762})
abnormal Counter({'1': 550})


In [8]:
# 8. processed 데이터를 train/val/test로 분할
# normal과 abnormal을 각각 70:20:10 비율로 나눈 뒤 합침

random.seed(42)

splits = {
    "train": [],
    "val": [],
    "test": []
}

for status in CLASS_MAP.keys():
    status_items = [item for item in processed_items if item["status"] == status]
    random.shuffle(status_items)

    n = len(status_items)
    n_train = int(n * 0.7)
    n_val = int(n * 0.2)

    splits["train"].extend(status_items[:n_train])
    splits["val"].extend(status_items[n_train:n_train + n_val])
    splits["test"].extend(status_items[n_train + n_val:])

print({split: len(items) for split, items in splits.items()})

for split in ["train", "val", "test"]:
    print(split, Counter([item["status"] for item in splits[split]]))

{'train': 771, 'val': 220, 'test': 112}
train Counter({'normal': 417, 'abnormal': 354})
val Counter({'normal': 119, 'abnormal': 101})
test Counter({'normal': 60, 'abnormal': 52})


In [9]:
# 9. split된 processed 데이터를 YOLO 학습용 폴더로 복사
# YOLO 폴더에서는 normal/abnormal 이미지가 섞여 있어도 됨
# class 구분은 txt 라벨의 첫 숫자 0/1로 처리

for split, split_items in splits.items():
    for item in split_items:
        img_src = item["image"]
        lbl_src = item["label"]

        img_dst = YOLO / "images" / split / img_src.name
        lbl_dst = YOLO / "labels" / split / lbl_src.name

        shutil.copy2(img_src, img_dst)
        shutil.copy2(lbl_src, lbl_dst)

print("YOLO 데이터셋 생성 완료")

YOLO 데이터셋 생성 완료


In [10]:
# 10. YOLO 폴더의 train/val/test 이미지 수와 라벨 수 확인
# images 수와 labels 수가 같아야 함

for split in ["train", "val", "test"]:
    img_count = len(list((YOLO / "images" / split).glob("*.jpg")))
    lbl_count = len(list((YOLO / "labels" / split).glob("*.txt")))
    print(split, "images:", img_count, "labels:", lbl_count)

train images: 771 labels: 771
val images: 220 labels: 220
test images: 112 labels: 112


In [11]:
# 11. YOLO 라벨의 class 분포 확인
# 0은 normal, 1은 abnormal
# train/val/test 모두 0과 1이 적절히 포함되어야 함

for split in ["train", "val", "test"]:
    counts = Counter()

    for txt in (YOLO / "labels" / split).glob("*.txt"):
        for line in txt.read_text().strip().splitlines():
            if line.strip():
                counts[line.split()[0]] += 1

    print(split, counts)

train Counter({'0': 534, '1': 382})
val Counter({'0': 152, '1': 106})
test Counter({'0': 76, '1': 62})
